## Example 1: Client construction and resource surface (offline)

In [ ]:
# Example 1: Client construction and resource surface (offline, no API key)
# mistralai 2 and later expose the client as mistralai.client.Mistral; `mistralai` itself is a namespace
# package with no flat `from mistralai import Mistral`.
from mistralai.client import Mistral, RetryConfig, VERSION, OPENAPI_DOC_VERSION

print("Mistral AI - client construction and resource surface")
print(f"   SDK VERSION={VERSION}  OpenAPI doc={OPENAPI_DOC_VERSION}")

# Constructing the client does NOT hit the network; a placeholder key is fine offline.
client = Mistral(api_key="demo_key_placeholder")
print(f"   client type: {type(client).__module__}.{type(client).__name__}")

# The client exposes API areas as attribute-accessed resource objects.
expected_resources = ["chat", "embeddings", "models", "agents", "files",
                      "fim", "ocr", "batch", "classifiers", "beta"]
present = [r for r in expected_resources if hasattr(client, r)]
missing = [r for r in expected_resources if not hasattr(client, r)]
assert not missing, f"missing resources: {missing}"
print("   resources present:")
for r in present:
    print(f"      - client.{r}: {type(getattr(client, r)).__name__}")

# RetryConfig is the retry setting (it replaces the 1.x dict config).
retry = RetryConfig("backoff", None, False)
print(f"   RetryConfig strategy: {retry.strategy}")
print("Example 1 completed")


## Example 2: Chat message model classes (offline)

In [ ]:
# Example 2: Chat message model classes (offline)
# The typed message classes live under mistralai.client.models, not the top level.
from mistralai.client.models import (
    UserMessage, SystemMessage, AssistantMessage, ToolMessage,
)
import pydantic

print("Mistral AI - chat message classes")

system_msg = SystemMessage(content="You are a helpful AI assistant.")
user_msg = UserMessage(content="What is Python?")
assistant_msg = AssistantMessage(content="Python is a programming language.")
tool_msg = ToolMessage(content="42", tool_call_id="call_abc")

# All message classes are pydantic models with a fixed .role discriminator.
for msg in (system_msg, user_msg, assistant_msg, tool_msg):
    assert isinstance(msg, pydantic.BaseModel)
    print(f"   {type(msg).__name__:16} role={msg.role!r}")

# A conversation is just an ordered list of these models.
conversation = [
    SystemMessage(content="You are a Python programming expert."),
    UserMessage(content="How do I read a CSV file?"),
    AssistantMessage(content="Use csv.reader or pandas.read_csv."),
]
print(f"   conversation has {len(conversation)} messages: "
      + " -> ".join(type(m).__name__.replace("Message", "") for m in conversation))

# model_dump() gives the exact JSON the API would receive.
print(f"   user_msg.model_dump(): {user_msg.model_dump()}")
print("Example 2 completed")


## Example 3: Embeddings resource and response model (offline)

In [ ]:
# Example 3: Embeddings resource + response model (offline)
import inspect
from mistralai.client import Mistral
from mistralai.client.models import EmbeddingResponse

print("Mistral AI - embeddings surface")

client = Mistral(api_key="demo_key_placeholder")

# The embeddings endpoint is client.embeddings.create(...). We inspect its
# signature offline rather than calling it (a real call needs an API key).
sig = inspect.signature(client.embeddings.create)
params = list(sig.parameters)
assert "model" in params and "inputs" in params, params
print(f"   client.embeddings.create params: {params[:6]}")

# EmbeddingResponse is the typed model a real call returns.
print(f"   EmbeddingResponse fields: {list(EmbeddingResponse.model_fields)}")

# Validate a response shaped like the API's reply, with synthetic vectors (no network), and compare its
# embeddings with cosine similarity in plain Python.
import math
import random

rng = random.Random(0)
vectors = [[rng.gauss(0, 1) for _ in range(1024)] for _ in range(3)]
response = EmbeddingResponse.model_validate({
    "id": "embd-local", "object": "list", "model": "mistral-embed",
    "usage": {"prompt_tokens": 3, "total_tokens": 3},
    "data": [{"object": "embedding", "embedding": v, "index": i} for i, v in enumerate(vectors)],
})
assert [item.index for item in response.data] == [0, 1, 2]
assert all(len(item.embedding) == 1024 for item in response.data)

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

embeddings = [item.embedding for item in response.data]
assert abs(cosine(embeddings[0], embeddings[0]) - 1.0) < 1e-9
print("   cosine similarity of the synthetic embeddings (real vectors come from the API):")
for i in range(len(embeddings)):
    for j in range(i + 1, len(embeddings)):
        similarity = cosine(embeddings[i], embeddings[j])
        # independent random vectors are close to orthogonal
        assert -0.2 < similarity < 0.2, similarity
        print(f"      vec{i+1} <-> vec{j+1}: {similarity:.3f}")
print("Example 3 completed")


## Example 4: Function calling with typed Tool and Function models (offline)

In [ ]:
# Example 4: Function calling with typed Tool / Function models (offline)
import inspect
from mistralai.client import Mistral
from mistralai.client.models import Tool, Function, ToolCall, FunctionCall

print("Mistral AI - function calling / tools")

# Tools are typed pydantic models, not raw dicts.
weather = Tool(function=Function(
    name="get_weather",
    description="Get the current weather for a city",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The city name"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
        },
        "required": ["city"],
    },
))
print(f"   Tool.type={weather.type!r}  function.name={weather.function.name!r}")
print(f"   serialized tool -> {weather.model_dump()['function']['name']}")

# client.chat.complete accepts a tools= argument (typed list); verify offline.
client = Mistral(api_key="demo_key_placeholder")
assert "tools" in inspect.signature(client.chat.complete).parameters
print("   client.chat.complete accepts tools=[...]")

# A model tool-call response is itself a typed ToolCall(FunctionCall).
tc = ToolCall(function=FunctionCall(name="get_weather", arguments='{"city": "Tokyo"}'))
print(f"   ToolCall.function.name={tc.function.name!r} args={tc.function.arguments!r}")

# The local function the app would run once the model asks for it.
def get_weather(city: str, unit: str = "celsius") -> str:
    return f"Weather in {city}: 25 {unit}"
print(f"   local execution: {get_weather('Tokyo')}")
print("Example 4 completed")


## Example 5: Models resource and typed model-card response (offline)

In [ ]:
# Example 5: Models resource + typed model-card response (offline)
import inspect
from mistralai.client import Mistral
from mistralai.client.models import BaseModelCard, ModelList

print("Mistral AI - models listing surface")

client = Mistral(api_key="demo_key_placeholder")

# The models endpoints are client.models.{list,retrieve,delete,...}. Verify the
# surface offline (a real client.models.list() call needs an API key).
for method in ("list", "retrieve", "delete"):
    assert hasattr(client.models, method), method
    print(f"   client.models.{method}: present")
print(f"   client.models.list params: {list(inspect.signature(client.models.list).parameters)}")

# ModelList is the typed envelope a real list() returns; BaseModelCard is each entry.
print(f"   ModelList fields: {list(ModelList.model_fields)}")
card_fields = BaseModelCard.model_fields
print(f"   BaseModelCard fields ({len(card_fields)}): {list(card_fields)[:8]}")

# Inspect which fields the decoder requires vs treats as optional (offline, no
# construction needed -- a live response supplies id/object/created/capabilities).
required = [n for n, f in card_fields.items() if f.is_required()]
optional = [n for n, f in card_fields.items() if not f.is_required()]
print(f"   BaseModelCard required: {required}")
print(f"   BaseModelCard optional (first 6): {optional[:6]}")
assert "id" in card_fields and "capabilities" in required
print("Example 5 completed")
